# Milestone 1 lab, notebook 3 of 7: The answer by hand

## Where we are

| Notebook | What you do |
|---|---|
| 1. The question and the source | Read the question. Meet the order system. Look at one order |
| 2. Read every order | Follow the pages of the API to the end |
| **3. The answer by hand** (you are here) | Work out the answer with plain Python. Meet the time zone trap |
| 4. Transform, then load | Put the orders in a warehouse table. Ask it one SQL question |
| 5. Every morning | Run it again the next day. Watch an old number change |
| 6. Why batch, and where it breaks | Measure how late the report is. Then watch the load fail |
| 7. The production version | The same logic as a tested program |

You have every order. Before any tool touches them, you answer the manager's question yourself, with plain Python you can check with a pencil.

That gives you an **answer key**. Every later version of the pipeline, in every milestone, is held up against it.

In [1]:
from lab_helpers import bars, get_all_orders, money, save_result, set_morning, show_doc

set_morning("2026-06-08")
orders, calls = get_all_orders()  # the finished loop from notebook 2
print(f"{len(orders)} orders in {calls} calls.")

It is now Monday 2026-06-08 07:00:00 in Arden.
226 orders in 10 calls.


## Step 1: A first try

It is Monday morning. "Yesterday" is Sunday 7 June. "The day before" is Saturday 6 June.

Every order has a `placed_at`, like `2026-06-07T04:58:12Z`. The first ten characters are a date. The easy idea: use them.

In [2]:
def count_day(day, which_day):
    """Count the orders of one day, and add up their money."""
    chosen = [order for order in orders if which_day(order) == day]
    return len(chosen), sum(order["total_cents"] for order in chosen)


def first_ten_characters(order):
    return order["placed_at"][:10]


for name, day in [("Sunday  ", "2026-06-07"), ("Saturday", "2026-06-06")]:
    count, cents = count_day(day, first_ten_characters)
    print(f"{name} {day}: {count} orders, {money(cents)}")

Sunday   2026-06-07: 33 orders, $810.37
Saturday 2026-06-06: 42 orders, $1,069.75


That ran with no error. Is it right?

## Step 2: Check it with a pencil

Never trust a number you cannot check. Take the smallest slice you can: **one city, one day**. Here is every order this first try counts as "Lakeshore, Sunday".

The second time column shows the same moment on the clock in Arden's cities.

In [3]:
from datetime import datetime
from zoneinfo import ZoneInfo

ARDEN = ZoneInfo("America/Chicago")  # the clock in Arden's three cities


def arden_time(utc_text):
    """A UTC time from the API, shown on Arden's clock."""
    return datetime.fromisoformat(utc_text.replace("Z", "+00:00")).astimezone(ARDEN)


lakeshore = [o for o in orders if o["city"] == "LKS" and first_ten_characters(o) == "2026-06-07"]

print("order   placed_at from the API     on Arden's clock        total")
for o in sorted(lakeshore, key=lambda o: o["placed_at"]):
    local = arden_time(o["placed_at"])
    print(
        f"{o['order_id']}    {o['placed_at']}   {local:%a %d %b, %H:%M}    {money(o['total_cents']):>9}"
    )

order   placed_at from the API     on Arden's clock        total
1190    2026-06-07T00:52:57Z   Sat 06 Jun, 19:52       $26.22
1197    2026-06-07T12:08:59Z   Sun 07 Jun, 07:08       $19.04
1198    2026-06-07T12:59:07Z   Sun 07 Jun, 07:59       $27.44
1203    2026-06-07T15:08:32Z   Sun 07 Jun, 10:08        $5.28
1211    2026-06-07T17:47:10Z   Sun 07 Jun, 12:47       $33.10
1215    2026-06-07T19:59:03Z   Sun 07 Jun, 14:59       $26.22
1219    2026-06-07T22:18:24Z   Sun 07 Jun, 17:18       $33.90


Look at the right-hand clock. The list is called "Sunday", but its first order was placed on **Saturday evening**. It does not belong here.

The API writes its times in **UTC**. Arden's cities are 5 hours behind UTC in June. So every order placed after 19:00 in Arden already has tomorrow's date in UTC.

One order shows it best.

In [4]:
late = next(o for o in orders if o["placed_at"] == "2026-06-07T04:58:12Z")

print("Order", late["order_id"], "in", late["city"])
print("  placed_at from the API:", late["placed_at"], " -> the text says Sunday 7 June")
print("  on Arden's clock:      ", arden_time(late["placed_at"]).strftime("%A %d %B, %H:%M:%S"))

Order 1196 in RVB
  placed_at from the API: 2026-06-07T04:58:12Z  -> the text says Sunday 7 June
  on Arden's clock:       Saturday 06 June, 23:58:12


A customer in Riverbend ordered ice cream, bananas and coffee at two minutes to midnight on **Saturday**. The first try counted it on Sunday.

## Step 3: What the business says

Which day is right is not a technical question. The business decides it.

In [5]:
show_doc("BRD.md", "5. Business rules")

**From `docs/BRD.md`**

> ## 5. Business rules
> 
> | Rule | Decision |
> |---|---|
> | Business day | The date the order was **placed**, on the clock in Arden's cities (`America/Chicago`) |
> | Which orders count | Every order that is not cancelled |
> | Money | The order's total, less any refund. Whole cents, US dollars |
> | Cities | Northgate (`NGT`), Riverbend (`RVB`), Lakeshore (`LKS`) |
> | When the report is made | 07:00, for the day before. The manager reads it at 08:00 |
> | "Since I last looked" | Since the report of the morning before |
> 
> The first rule hides a trap. An order placed at 23:58 on Saturday belongs to **Saturday**. The order system writes its times in UTC, where that moment is already Sunday.

The first row is the rule. So write it as code. The business day is the date of `placed_at`, **after** it is moved to Arden's clock.

In [6]:
def business_day(order):
    """The day an order belongs to: the date it was placed, on Arden's clock."""
    return arden_time(order["placed_at"]).date().isoformat()


print("Order", late["order_id"], "belongs to", business_day(late))

Order 1196 belongs to 2026-06-06


## Step 4: The second try

The same count, with the right day.

In [7]:
right, wrong = {}, {}
for name, day in [("Sunday 06-07", "2026-06-07"), ("Saturday 06-06", "2026-06-06")]:
    right[name] = count_day(day, business_day)
    wrong[name] = count_day(day, first_ten_characters)

print(f"{'':<16}{'UTC date (wrong)':>24}{'Arden clock (right)':>26}")
for name in right:
    w, r = wrong[name], right[name]
    print(f"{name:<16}{w[0]:>10} orders {money(w[1]):>9}{r[0]:>12} orders {money(r[1]):>9}")

                        UTC date (wrong)       Arden clock (right)
Sunday 06-07            33 orders   $810.37          30 orders   $807.11
Saturday 06-06          42 orders $1,069.75          33 orders   $786.71


In [8]:
print("Orders counted on Saturday 6 June")
bars(
    {
        "UTC date (wrong)": wrong["Saturday 06-06"][0],
        "Arden clock (right)": right["Saturday 06-06"][0],
    },
    unit=" orders",
)

Orders counted on Saturday 6 June
UTC date (wrong)     ████████████████████████████████████████ 42 orders
Arden clock (right)  ███████████████████████████████ 33 orders


Saturday was wrong by nine orders and almost $300. Nothing failed. No error, no warning. A wrong number with no error is worse than a crash, because people trust it.

Now the pencil check again. Lakeshore, Sunday, with the right day:

In [9]:
lakeshore = [o for o in orders if o["city"] == "LKS" and business_day(o) == "2026-06-07"]

for o in sorted(lakeshore, key=lambda o: o["placed_at"]):
    print(f"{o['order_id']}   {arden_time(o['placed_at']):%a %H:%M}   {o['total_cents']:>6}")
print("                    ------")
print(
    f"{len(lakeshore)} orders           {sum(o['total_cents'] for o in lakeshore):>6} cents = {money(sum(o['total_cents'] for o in lakeshore))}"
)

1197   Sun 07:08     1904
1198   Sun 07:59     2744
1203   Sun 10:08      528
1211   Sun 12:47     3310
1215   Sun 14:59     2622
1219   Sun 17:18     3390
                    ------
6 orders            14498 cents = $144.98


Add those six numbers yourself. You should get 14,498.

## Step 5: Two new words

| Word | Meaning | At Arden |
|---|---|---|
| **Event time** | "Event time is when the thing happened in the real world." | `placed_at`: when the customer ordered |
| **Grain** | "The grain of a result is what one row means." | In our list, one row is one order |

The definitions are from the D8Loop chapters [Batch vs Streaming vs Micro-Batch](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/batch-vs-streaming-vs-micro-batch) and [Transformation Fundamentals](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/transformation-fundamentals).

Event time has a rule of thumb: **use event time for business numbers**. The manager asks about the day customers ordered, not the day a program happened to run.

### Why grain matters: compare it with its opposite

Our list has one row per **order**. The API also gives the **items** inside each order. Suppose you make one row per item, and each row carries its order's total. Then add up the totals.

In [10]:
sunday = [o for o in orders if business_day(o) == "2026-06-07"]

one_row_per_order = sum(o["total_cents"] for o in sunday)
one_row_per_item = sum(o["total_cents"] for o in sunday for item in o["items"])

print(f"Rows at order grain: {len(sunday):>4}    sum of totals: {money(one_row_per_order)}")
print(
    f"Rows at item grain:  {sum(len(o['items']) for o in sunday):>4}    sum of totals: {money(one_row_per_item)}"
)

Rows at order grain:   30    sum of totals: $807.11
Rows at item grain:   110    sum of totals: $3,198.77


The second number is almost four times too big. An order with four items was counted four times.

Always say what one row means before you add anything up.

## Step 6: One more rule, cancelled orders

The business rules say a cancelled order does not count. Neither weekend day has one. But one earlier day does.

In [11]:
cancelled = [o for o in orders if o["status"] == "cancelled"]
for o in cancelled:
    print(
        f"Order {o['order_id']}, placed on {business_day(o)}, total {money(o['total_cents'])}, status {o['status']}"
    )

Order 1091, placed on 2026-06-03, total $20.56, status cancelled


If you forgot this rule, Wednesday 3 June would show one order and $20.56 too much. The answer key below follows the rule.

## Step 7: The answer key

Now work out everything the manager asked for, with plain loops. No pandas, no SQL.

In [12]:
def one_day(day):
    """Orders and money for one business day, in total and for each city."""
    result = {"order_date": day, "orders": 0, "revenue_cents": 0, "by_city": {}}
    for o in orders:
        if business_day(o) != day or o["status"] == "cancelled":
            continue
        money_in = o["total_cents"] - o["refunded_cents"]
        city = result["by_city"].setdefault(o["city"], {"orders": 0, "revenue_cents": 0})
        city["orders"] += 1
        city["revenue_cents"] += money_in
        result["orders"] += 1
        result["revenue_cents"] += money_in
    return result


yesterday = one_day("2026-06-07")
day_before = one_day("2026-06-06")

answer_key = {
    "run_date": "2026-06-08",
    "yesterday": yesterday,
    "day_before": day_before,
    "change": {
        "orders": yesterday["orders"] - day_before["orders"],
        "revenue_cents": yesterday["revenue_cents"] - day_before["revenue_cents"],
    },
}

print(
    f"Yesterday   Sun 2026-06-07   {yesterday['orders']} orders   {money(yesterday['revenue_cents'])}"
)
print(
    f"Day before  Sat 2026-06-06   {day_before['orders']} orders   {money(day_before['revenue_cents'])}"
)
print(
    f"Change                       {answer_key['change']['orders']:+d} orders   {money(answer_key['change']['revenue_cents'])}"
)
print()
for city, row in sorted(yesterday["by_city"].items()):
    print(f"  {city}   {row['orders']:>2} orders   {money(row['revenue_cents']):>8}")

Yesterday   Sun 2026-06-07   30 orders   $807.11
Day before  Sat 2026-06-06   33 orders   $786.71
Change                       -3 orders   $20.40

  LKS    6 orders    $144.98
  NGT   10 orders    $355.26
  RVB   14 orders    $306.87


In [13]:
save_result("answer_key", answer_key)

Saved results/answer_key.json


That file is the answer key. Notebook 4 checks SQL against it. A test checks the production code against it.

The fourth part of the question, "has any earlier day changed since I last looked?", needs a second morning. It waits for notebook 5.

## What you now know

- `placed_at` is in UTC. Arden's day is decided on Arden's clock. Mixing them moves every evening order to the wrong day.
- Event time is when the thing happened. Business numbers use event time.
- The grain is what one row means. Adding up at the wrong grain counts things twice.
- The answer: Sunday 30 orders and $807.11. Saturday 33 orders and $786.71.

**Say it in an interview:** "The source stored times in UTC, but the business day was local. An order placed at 23:58 on Saturday was Sunday in UTC. Grouping by the UTC date put nine extra orders on Saturday with no error, so I convert event time to the business time zone before I take the date."

Next: [notebook 4](04_transform_then_load.ipynb) puts these orders in a warehouse and gets the same answer with SQL.